# Chapter 12 &mdash; A Tale of Three Parsers, and Why Textual Syntax Still Rules

**Concept 11 of the Chapter 12 decomposition:** *A Tale of Three Parsers, and Why Textual Syntax Still Rules*

`re2nfa`, the derivative mini-compiler and `md2mc` &mdash; three real parsers you have already used.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-Tale-Of-Three-Parsers/Concept-Tale-Of-Three-Parsers.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


You have used three parsers in this book, each a working example of the theory:

* **`re2nfa`** (Chapter 8) &mdash; lexer plus LALR parser; semantic actions build NFA
  fragments;
* **`re2ast`** (Chapter 10) &mdash; the same shape, but the actions build an **AST**, and
  the grammar has two extra operators (`!`, `&`);
* **`md2mc`** &mdash; a line-oriented parser for machine descriptions, with its own
  conventions (`I`/`F` prefixes, `!!` comments).

And the closing observation: **textual syntax still rules.** Graphical machine editors
exist, but text diffs, greps, version-controls, generates and reviews. The markdown
formats in this book are not a limitation; they are why the machines are maintainable.

## 2. Definitions

### The three parsers, side by side

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
import jove.Def_RE2NFA as P1
import jove.Def_rederiv as P2

def tokens_of(mod):
    return sorted(getattr(mod, 'tokens', []))

def productions_of(mod):
    return sorted((n, (getattr(mod, n).__doc__ or '').strip())
                  for n in dir(mod) if n.startswith('p_') and n != 'p_error')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### What each one produces

In [ ]:
OUTPUTS = [('re2nfa', 'an NFA (Thompson fragments glued with epsilon edges)'),
           ('re2ast', 'an AST of nested tuples, for the derivative matcher'),
           ('md2mc',  'a machine dictionary: DFA, NFA, PDA or TM')]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;10.&nbsp;Disambiguation Measured: 1 Parse versus 36](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-Disambiguation-Measured/Concept-Disambiguation-Measured.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;1.&nbsp;Turing's Definition of Computation, and the Entscheidungsproblem](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Turings-Definition/Concept-Turings-Definition.ipynb)&nbsp;&rarr;

---

## 3. Tests

Two of them share a token set; one adds operators.

In [ ]:
print("re2nfa  tokens :", tokens_of(P1))
print("rederiv tokens :", tokens_of(P2))
extra = set(tokens_of(P2)) - set(tokens_of(P1))
print("extra in rederiv :", sorted(extra))
assert extra == {'AND', 'NOT'}

Same grammar shape, different **semantic actions**.

In [ ]:
for n, doc in productions_of(P1)[:4]:
    print("  re2nfa  %-26s %s" % (n, doc))
print()
for n, doc in productions_of(P2)[:4]:
    print("  rederiv %-26s %s" % (n, doc))
print("\nThe productions match almost line for line; only the actions differ.")

Each produces a different object from the same input.

In [ ]:
r = "(0+1)*1"
N = re2nfa(r)
A = re2ast(r)[0]
print("re2nfa  -> NFA with %d states" % len(N["Q"]))
print("re2ast  -> AST %s" % (A,))
assert 'Q0' in N and isinstance(A, tuple)

`md2mc` is the third, and it parses **four** machine types from one notation.

In [ ]:
kinds = {'DFA': 'DFA\nIF : 0 -> IF\n',
         'NFA': "NFA\nI : 0 -> F\nF : '' -> I\n",
         'PDA': "PDA\nI : a , # ; A# -> I\nI : '' , # ; # -> F\n",
         'TM' : "TM\nI : 0 ; 1 , R -> F\n"}
for k, src in kinds.items():
    m = md2mc(src)
    print("  %-4s -> dict with keys %s" % (k, sorted(m.keys())))
assert 'Gamma' in md2mc(kinds['PDA'])

And they all agree where they overlap.

In [ ]:
from itertools import product
D1 = min_dfa(nfa2dfa(re2nfa("(0+1)*01")))
# the derivative matcher, in brief
EPS, PHI = ('@', '@'), ('phi', 'phi')
def nullable(E):
    t = E[0]
    return {'@': True, 'phi': False, 'str': False, '*': True}.get(t) if t in \
        ('@', 'phi', 'str', '*') else (
        nullable(E[1][0]) or nullable(E[1][1]) if t == '+' else
        nullable(E[1][0]) and nullable(E[1][1]) if t in ('.', '&') else
        not nullable(E[1]))
def dv(c, E):
    t = E[0]
    if t in ('@', 'phi'): return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+': return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&': return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*': return ('.', (dv(c, E[1]), E))
    if t == '!': return ('!', dv(c, E[1]))
    E1, E2 = E[1]
    left = ('.', (dv(c, E1), E2))
    return ('+', (left, dv(c, E2))) if nullable(E1) else left
def matches(s, E):
    for ch in s: E = dv(ch, E)
    return nullable(E)

A = re2ast("(0+1)*01")[0]
strs = [''.join(p) for k in range(9) for p in product('01', repeat=k)]
assert all(matches(s, A) == accepts_dfa(D1, s) for s in strs)
print("the NFA route and the AST route agree on all %d strings" % len(strs))

Why textual syntax still rules.

In [ ]:
for why in ["it diffs -- you can see what changed between two machines",
            "it greps -- you can find every transition on a symbol",
            "it version-controls -- machines live in git like code",
            "it generates -- programs can emit it (cfg2pda does)",
            "it comments -- '!!' explains WHY a transition is there",
            "it reviews -- a colleague can read it without a tool"]:
    print("  *", why)

## 4. Exercises


1. Add a `?` operator to `re2ast`. Which production and which AST node?
2. Write a program that emits `md2mc` text for a random DFA.
3. What would a graphical editor have to offer to beat text? Has anything?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter12/Concept-Tale-Of-Three-Parsers')